In [ ]:
import os, time, joblib, warnings
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd

# RDKit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys

# Turning off Deprecation Warning from RDKit
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore")

# Sklearn
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_regression, VarianceThreshold

# Models
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
#1. PATHS + PARAMETERS
INPUT_CSV = r"D:/Machine Learning/Assay_ThT/Compiled_data.csv"
PREDICT_CSV = r"D:/Machine Learning/Assay_ThT/daxy_PTPyr.csv"
OUT_DIR = Path(r"D:/Machine Learning/Assay_ThT_2311/Final_Production_Model_PTPyr")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IC50_CAP = 40000.0
FP_BITS = 2048
FP_RADIUS = 2
BEST_K_DESC = 50
BEST_K_FP = 50
RANDOM_SEED = 42

#2. FEATURES + DATA PREPROCESSING (RDKIT + MACCS + ECFP4)
def load_data_robust(filepath):
    """Read files with any encoding"""
    for enc in ['utf-8', 'latin-1', 'cp1252', 'utf-16']:
        try:
            return pd.read_csv(filepath, encoding=enc)
        except Exception:
            continue
    try:
        return pd.read_excel(filepath)
    except Exception:
        return None

def standardize_smiles(smi):
    try:
        if pd.isna(smi) or str(smi).strip() == '':
            return None
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return None
        frags = Chem.GetMolFrags(mol, asMols=True)
        if len(frags) > 1:
            mol = sorted(frags, key=lambda m: m.GetNumAtoms(), reverse=True)[0]
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def compute_features_inference(df_raw, training_desc_names=None):
    """
    Hybrid Features which include:
    - 2D RDKit Descriptors.
    - MACCS Keys (167 bits) with functional groups.
    - Morgan Fingerprints ECFP4 (2048 bits).
    """
    print(f">>> Computing hybrid features (RDKit + MACCS + ECFP4) for {len(df_raw)} rows...")
    desc_dict = dict(Descriptors.descList)
    target_names = training_desc_names if training_desc_names is not None else [x[0] for x in Descriptors.descList]

    data_list = []
    valid_indices = []
    rdkit_fps = []

    mols = [Chem.MolFromSmiles(str(s)) if pd.notna(s) else None for s in df_raw['std_smiles']]

    for idx, (orig_idx, _) in enumerate(df_raw.iterrows()):
        mol = mols[idx]
        if mol is None:
            continue

        #2a. RDKit 2D Descriptors
        desc_vals = []
        for name in target_names:
            func = desc_dict.get(name)
            try:
                v = float(func(mol)) if func is not None else np.nan
                if not np.isfinite(v) or abs(v) > 1e6:
                    v = np.nan
                desc_vals.append(v)
            except Exception:
                desc_vals.append(np.nan)

        #2b. MACCS Fingerprints (167 bits)
        maccs_obj = MACCSkeys.GenMACCSKeys(mol)
        maccs_arr = np.array(maccs_obj, dtype=np.float64)

        #2c. Morgan Fingerprints (2048 bits)
        fp_obj = AllChem.GetMorganFingerprintAsBitVect(mol, FP_RADIUS, FP_BITS)
        fp_arr = np.array(fp_obj, dtype=np.float64)

        data_list.append(np.concatenate([desc_vals, maccs_arr, fp_arr]))
        valid_indices.append(orig_idx)
        rdkit_fps.append(fp_obj)

    X = np.array(data_list, dtype=np.float64)

    if training_desc_names is None:
        y_p = df_raw.loc[valid_indices, 'pIC50'].values
        return X, y_p, target_names, valid_indices, rdkit_fps
    else:
        return X, valid_indices, rdkit_fps

#3. PIPELINE
def get_pipeline(n_desc_cols, model_type='xgb', k_desc=120, k_fp=350, random_state=42):
    mi_fixed = partial(mutual_info_regression, random_state=random_state)

    desc_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('var', VarianceThreshold(threshold=0.0)),
        ('scaler', RobustScaler()),
        ('selector', SelectKBest(score_func=mi_fixed, k=k_desc))
    ])

    fp_pipe = Pipeline([
        ('var', VarianceThreshold(threshold=0.01)),
        ('selector', SelectKBest(score_func=mi_fixed, k=k_fp))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('desc_proc', desc_pipe, slice(0, n_desc_cols)),
        ('fp_proc', fp_pipe, slice(n_desc_cols, None))
    ])

    if model_type == 'xgb':
        model = XGBRegressor(
            n_estimators=500, max_depth=4, learning_rate=0.02,
            min_child_weight=2, reg_alpha=0.2, reg_lambda=1.5,
            subsample=0.8, colsample_bytree=0.7, n_jobs=-1, random_state=random_state
        )
    elif model_type == 'rf':
        model = RandomForestRegressor(
            n_estimators=400, max_depth=12, min_samples_leaf=2,
            max_features=0.5, n_jobs=-1, random_state=random_state
        )
    elif model_type == 'et':
        model = ExtraTreesRegressor(
            n_estimators=400, max_depth=14, min_samples_leaf=2,
            max_features=0.5, n_jobs=-1, random_state=random_state
        )
    else:
        model = Pipeline([
            ('scale', StandardScaler()),
            ('svm', SVR(C=2.5, epsilon=0.05, kernel='rbf', gamma='scale'))
        ])

    return Pipeline([('preprocessor', preprocessor), ('model', model)])

def build_applicability_domain(X_train, y_train, oof_preds, train_fps, train_smiles, n_pca_ad=20):
    ad_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('var', VarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_pca_ad, random_state=RANDOM_SEED))
    ])
    Z_train = ad_pipe.fit_transform(X_train)
    n_samples, p_dim = Z_train.shape
    mean_z = Z_train.mean(axis=0)
    Z_train_c = Z_train - mean_z
    inv_gram = np.linalg.pinv(Z_train_c.T @ Z_train_c)
    h_train = (1.0 / n_samples) + np.sum((Z_train_c @ inv_gram) * Z_train_c, axis=1)
    h_star = 3.0 * (p_dim + 1) / n_samples

    return {
        'ad_pipe': ad_pipe,
        'mean_z': mean_z,
        'inv_gram': inv_gram,
        'n_train': n_samples,
        'p_dim': p_dim,
        'h_star': h_star,
        'h_train': h_train,
        'train_fps': train_fps,
        'train_smiles': list(train_smiles),
        'train_y': y_train
    }

def evaluate_new_applicability_domain(X_new, new_fps, ad_bundle):
    Z_new = ad_bundle['ad_pipe'].transform(X_new)
    Z_new_c = Z_new - ad_bundle['mean_z']
    h_new = (1.0 / ad_bundle['n_train']) + np.sum((Z_new_c @ ad_bundle['inv_gram']) * Z_new_c, axis=1)
    h_star = ad_bundle['h_star']
    train_fps = ad_bundle['train_fps']
    train_smiles = ad_bundle['train_smiles']
    train_y = ad_bundle['train_y']

    max_tanimoto = []
    top5_tanimoto = []
    nearest_smi = []
    nearest_pic50 = []

    for fp in new_fps:
        sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, train_fps))
        best_idx = int(np.argmax(sims))
        top5_vals = np.sort(sims)[-5:]

        max_tanimoto.append(float(sims[best_idx]))
        top5_tanimoto.append(float(np.mean(top5_vals)))
        nearest_smi.append(train_smiles[best_idx])
        nearest_pic50.append(float(train_y[best_idx]))

    max_tanimoto = np.array(max_tanimoto)
    top5_tanimoto = np.array(top5_tanimoto)

    ad_status = []
    for h_val, s_max in zip(h_new, max_tanimoto):
        if (h_val <= h_star) and (s_max >= 0.35):
            ad_status.append("High Confidence (In-AD)")
        elif (h_val <= h_star * 1.5) and (s_max >= 0.25):
            ad_status.append("Moderate Confidence (Borderline)")
        else:
            ad_status.append("Low Confidence (Out-of-AD)")

    return {
        'Leverage_h': h_new,
        'Leverage_Threshold_h_star': np.full(len(h_new), h_star),
        'Max_Tanimoto_Train': max_tanimoto,
        'Top5_Mean_Tanimoto': top5_tanimoto,
        'Nearest_Train_SMILES': nearest_smi,
        'Nearest_Train_pIC50': nearest_pic50,
        'AD_Reliability': ad_status
    }

def train_final_stacking(X, y, n_desc, train_fps, train_smiles):
    print("\n" + "="*60)
    print("  TRAINING OPTIMIZED NON-LINEAR STACKING MODEL")
    print("="*60)

    p_xgb = get_pipeline(n_desc, 'xgb', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_rf = get_pipeline(n_desc, 'rf', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_et = get_pipeline(n_desc, 'et', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_svr = get_pipeline(n_desc, 'svr', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    print("Generating 5-fold OOF predictions for Base Models (XGB, RF, ExtraTrees, SVR)...")

    o_xgb = cross_val_predict(p_xgb, X, y, cv=cv, n_jobs=-1)
    o_rf = cross_val_predict(p_rf, X, y, cv=cv, n_jobs=-1)
    o_et = cross_val_predict(p_et, X, y, cv=cv, n_jobs=-1)
    o_svr = cross_val_predict(p_svr, X, y, cv=cv, n_jobs=-1)

    X_meta = np.column_stack([o_xgb, o_rf, o_et, o_svr])

    meta_model = Ridge(alpha=5.0, positive=True, random_state=RANDOM_SEED)
    oof_stacking = cross_val_predict(meta_model, X_meta, y, cv=cv)

    q2_cv = r2_score(y, oof_stacking)
    rmse_cv = np.sqrt(mean_squared_error(y, oof_stacking))
    mae_cv = mean_absolute_error(y, oof_stacking)

    print(f"Base OOF R2 -> XGB: {r2_score(y, o_xgb):.4f} | RF: {r2_score(y, o_rf):.4f} | ET: {r2_score(y, o_et):.4f} | SVR: {r2_score(y, o_svr):.4f}")
    print(f"Stacking Nested OOF Q2_CV: {q2_cv:.4f} | RMSE_CV: {rmse_cv:.4f} | MAE_CV: {mae_cv:.4f}")

    print("Fitting Meta-Learner and Refitting Base Models on 100% dataset...")
    meta_model.fit(X_meta, y)
    print(f"Meta-Learner Weights (XGB, RF, ET, SVR): {np.round(meta_model.coef_, 4)}")

    p_xgb.fit(X, y)
    p_rf.fit(X, y)
    p_et.fit(X, y)
    p_svr.fit(X, y)

    ad_bundle = build_applicability_domain(X, y, oof_stacking, train_fps, train_smiles)

    return {
        'xgb': p_xgb, 'rf': p_rf, 'et': p_et, 'svr': p_svr,
        'meta': meta_model,
        'n_desc': n_desc,
        'oof_preds': oof_stacking,
        'q2_cv': q2_cv,
        'rmse_cv': rmse_cv,
        'ad_bundle': ad_bundle
    }

In [ ]:
#4. MAIN WORKFLOW
def main():
    print(f">>> Loading training data: {INPUT_CSV}")
    df_train = load_data_robust(INPUT_CSV)
    if df_train is None:
        print("Cannot find Training set. Check INPUT_CSV!")
        return

    ic50_col = [c for c in df_train.columns if 'ic50' in c.lower() and 'pic50' not in c.lower()][0]
    smi_col_train = next((c for c in df_train.columns if 'smile' in c.lower()), 'Smiles')

    df_train['capped_ic50'] = pd.to_numeric(df_train[ic50_col], errors='coerce').clip(upper=IC50_CAP)
    df_train['pIC50'] = 9.0 - np.log10(np.maximum(df_train['capped_ic50'], 1.0))
    df_train['std_smiles'] = df_train[smi_col_train].apply(standardize_smiles)
    df_train = df_train.dropna(subset=['std_smiles', 'pIC50']).reset_index(drop=True)

    X_train, y_train, desc_names, valid_train_idx, train_fps = compute_features_inference(df_train)
    train_smiles = df_train.loc[valid_train_idx, 'std_smiles'].values
    n_desc = len(desc_names)

    model_bundle = train_final_stacking(X_train, y_train, n_desc, train_fps, train_smiles)
    model_bundle['desc_names'] = desc_names

    model_path = OUT_DIR / "Stacking_SVR_Final.pkl"
    joblib.dump(model_bundle, model_path)
    print(f"\n Saved production model bundle to: {model_path}")

    # Prediction:
    print("\n" + "="*60)
    print("  PREDICTING NEW COMPOUNDS (DAXY) - NO SORTING (EXACT INPUT ORDER)")
    print("="*60)

    if not os.path.exists(PREDICT_CSV):
        print(f"Cannot find prediction file at: {PREDICT_CSV}!")
        return

    df_new = load_data_robust(PREDICT_CSV)
    smi_col_new = next((c for c in df_new.columns if 'smile' in c.lower()), None)
    if not smi_col_new:
        print("Cannot find SMILES row!")
        return

    print(f">>> Found {len(df_new)} rows in {PREDICT_CSV}")
    df_out = df_new.copy()
    df_out['Original_Row_ID'] = np.arange(1, len(df_out) + 1)
    df_out['std_smiles'] = df_out[smi_col_new].apply(standardize_smiles)

    X_new, valid_new_idx, new_fps = compute_features_inference(df_out, training_desc_names=desc_names)

    print("Running ensemble predictions!")
    p1 = model_bundle['xgb'].predict(X_new)
    p2 = model_bundle['rf'].predict(X_new)
    p3 = model_bundle['et'].predict(X_new)
    p4 = model_bundle['svr'].predict(X_new)

    X_meta_new = np.column_stack([p1, p2, p3, p4])
    final_pred_pIC50 = model_bundle['meta'].predict(X_meta_new)
    pred_std = np.std(X_meta_new, axis=1, ddof=1)
    final_pred_IC50_nM = 10 ** (9.0 - final_pred_pIC50)

    ad_results = evaluate_new_applicability_domain(X_new, new_fps, model_bundle['ad_bundle'])

    # df_out accor to index (valid_new_idx):
    df_out.loc[valid_new_idx, 'Pred_XGB'] = p1
    df_out.loc[valid_new_idx, 'Pred_RF'] = p2
    df_out.loc[valid_new_idx, 'Pred_ET'] = p3
    df_out.loc[valid_new_idx, 'Pred_SVR'] = p4
    df_out.loc[valid_new_idx, 'Predicted_pIC50'] = final_pred_pIC50
    df_out.loc[valid_new_idx, 'Ensemble_Pred_Std'] = pred_std
    df_out.loc[valid_new_idx, 'Predicted_IC50_nM'] = final_pred_IC50_nM
    df_out.loc[valid_new_idx, 'Predicted_IC50_uM'] = final_pred_IC50_nM / 1000.0

    for k, v in ad_results.items():
        df_out.loc[valid_new_idx, k] = v

    # df_out to file:
    out_file_main = OUT_DIR / "Daxy_Predictions_Stacking_with_AD.csv"
    out_file_orig = OUT_DIR / "Daxy_Predictions_Stacking_OriginalOrder.csv"
    df_out.to_csv(out_file_main, index=False, encoding='utf-8-sig')
    df_out.to_csv(out_file_orig, index=False, encoding='utf-8-sig')


    print(f"\n Done. Save at:")
    print(f"1. {out_file_main}")
    print(f"2. {out_file_orig}")
    cols_show = ['Original_Row_ID', smi_col_new, 'Predicted_pIC50', 'Predicted_IC50_uM', 'Ensemble_Pred_Std', 'AD_Reliability']
    print("\n10 first rows of daxy_KL.csv:")
    print(df_out[cols_show].head(10).to_string(index=False))

if __name__ == "__main__":
    main()